# Chat with a PDF (RAG, fully local)
A conversational chatbot that answers questions about a PDF using **retrieval-augmented generation**. Everything runs locally with open-source models through [Ollama](https://ollama.com): no API keys, and your documents never leave your machine.

**Prerequisites:** install Ollama and run `ollama pull llama3:8b`.

## 1. Load the PDF and split it into chunks
Chunks of 512 characters with 50 characters of overlap, so sentences at the boundaries aren't lost.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load PDF
PDF_PATH = "document.pdf"  # put any PDF in this folder and set its name here
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

## 2. Embed the chunks and store them in Chroma
Each chunk is turned into a vector with Ollama embeddings and stored in a persistent Chroma database (`./chroma_db`). This is the slowest step, and it only needs to run once per document.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

# Initialize embeddings
embeddings = OllamaEmbeddings(model="llama3:8b")

# Store in Chroma
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
vector_store.persist()

## 3. Build the conversational retrieval chain
The retriever fetches the 3 most relevant chunks for each question, and conversation memory lets you ask follow-up questions.

In [ ]:
from langchain_community.llms import Ollama
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

# Initialize LLM
llm = Ollama(model="llama3:8b")

# Set up memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Build conversational chain
chatbot = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

## 4. Ask questions

In [ ]:
query = "What is the main topic of the PDF?"
response = chatbot.invoke({"question": query})
print(response["answer"])